# Pipeline : Generate data files and update pathway model from snapshot

Run this notebook top-to-bottom whenever the snapshot model or prospective CSVs change.
All steps are idempotent: re-running is safe.

| Step | What it does | Writes to |
|------|-------------|----------|
| 1 | Runs snapshot model, applies prospective ratios, writes dat files directly to shared/data | `shared/data/Techs/`, `shared/data/EUD/`, `shared/data/Shares/` |
| 3 | Merges per-year files into AMPL matrix format | `shared/data/Techs/out_techs.dat` |
| 3b | Regenerates remaining_years from out_techs lifetimes | `projects/pathway/model/PES_data_remaining*.dat` |
| 4 | Regenerates AGE set from out_techs lifetimes | `projects/pathway/model/PES_data_set_AGE_2020.dat` |
| 5 | Regenerates decom_allowed from out_techs | `projects/pathway/model/PES_data_decom_allowed_2020.dat` |
| 6 | Regenerates years_active from out_techs lifetimes | `projects/pathway/model/PES_data_years_active.dat` |

## Configuration

In [ ]:
from pathlib import Path
import sys, shutil
from datetime import datetime


UTIL_DIR   = Path.cwd()                              # shared/utilities/
SHARED_DIR = UTIL_DIR.parent                         # shared/
REPO_ROOT  = SHARED_DIR.parent                       # repo_root/


# Input directories
SNAPSHOT_DIR    = UTIL_DIR / "ES_Snapshot"           # shared/utilities/ES_Snapshot
PATHWAY_DIR     = REPO_ROOT / "projects" / "pathway" / "model"
PROSPECTIVE_DIR = UTIL_DIR / "Prospective_parameters"
SCRIPTS_DIR     = UTIL_DIR / "scripts"               # shared/utilities/scripts

# Shared data directories (Step 1 writes directly here)
TECHS_DIR  = REPO_ROOT / "shared" / "data" / "Techs"
EUD_DIR    = REPO_ROOT / "shared" / "data" / "EUD"
SHARES_DIR = REPO_ROOT / "shared" / "data" / "Shares"

# Pathway model output files
AGE_FILE          = PATHWAY_DIR / "PES_data_set_AGE_2020.dat"
DECOM_FILE        = PATHWAY_DIR / "PES_data_decom_allowed_2020.dat"
REMAINING_FILE    = PATHWAY_DIR / "PES_data_remaining.dat"
REMAINING_WND     = PATHWAY_DIR / "PES_data_remaining_wnd.dat"
YEARS_ACTIVE_FILE = PATHWAY_DIR / "PES_data_years_active.dat"

SCENARIO    = "Current Measure"
YEAR_LABELS = ["2020", "2025", "2030", "2035", "2040", "2045", "2050"]

# Validation dat files that fix installed capacities in each calibration year.
# 2021 file -> snapshot calibrated to 2020 data  (used for QC_techs_2020 / QC_eud_2020)
# 2023 file -> snapshot calibrated to 2023 data  (used for QC_techs_2025-2050 / QC_eud_2025-2050)
VALIDATION_2021 = "QC_validation_2021.dat"
VALIDATION_2023 = "QC_validation_2023.dat"

# Flags
RUN_PROSPECTIVE = True   # Step 1: run snapshot model + write per-year dat files

sys.path.insert(0, str(SCRIPTS_DIR))
print(f"Root: {SHARED_DIR}")
print("Config loaded.")

## Step 1 : Generate prospective dat files

Runs the snapshot model, applies prospective ratios, and writes per-year
`QC_techs_<year>.dat` and `QC_eud_<year>.dat` files to
`Prospective_parameters/Outputs/dat_files/`.

Set `RUN_PROSPECTIVE = False` to skip if the files are already up to date.

In [ ]:
if RUN_PROSPECTIVE:
    import generate_dat_files
    print('Generating prospective dat files (runs snapshot model, may take a few minutes) ...')
    generate_dat_files.run(
        snapshot_dir        = SNAPSHOT_DIR,
        prospective_dir     = PROSPECTIVE_DIR,
        output_techs_dir    = TECHS_DIR,
        output_eud_dir      = EUD_DIR,
        output_shares_dir   = SHARES_DIR,
        scenario            = SCENARIO,
        validation_2021     = VALIDATION_2021,
        validation_2023     = VALIDATION_2023,
    )
else:
    print('Skipping Step 1 (RUN_PROSPECTIVE = False).')

In [ ]:
print('Prospective tech files (shared/data/Techs):')
any_missing = False
for yr in YEAR_LABELS:
    path = TECHS_DIR / f'QC_techs_{yr}.dat'
    if path.exists():
        mtime = datetime.fromtimestamp(path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        size  = path.stat().st_size
        print(f'  QC_techs_{yr}.dat   {mtime}   {size:,} bytes')
    else:
        print(f'  QC_techs_{yr}.dat   MISSING')
        any_missing = True

if any_missing:
    raise FileNotFoundError('Some prospective files are missing : run with RUN_PROSPECTIVE = True.')

## Step 3 : Write matrix dat files

Reads all per-year `QC_techs_<year>.dat` from `ES_Transition_QC_2/Techs/` and
`QC_eud_<year>.dat` from `ES_Transition_QC_2/EUD/`, then rewrites them in
AMPL matrix format as `out_techs.dat` and `out_eud.dat`.

In [6]:
import importlib
import write_matrix_dat
importlib.reload(write_matrix_dat)

print('Writing matrix dat files ...')
write_matrix_dat.run(TECHS_DIR, EUD_DIR, shares_dir=SHARES_DIR)

Writing matrix dat files ...
Loading techs from c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs ...
Loading EUD from c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\EUD ...
  Saved: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs\out_techs.dat
  Saved: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs\out_fmax_block.dat  (0 blocking entries)
  Saved: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\EUD\out_eud.dat
Loading shares from c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Shares ...
  Saved: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Shares\out_shares.dat
Done.


## Step 3b : Regenerate remaining_years files

Reads lifetimes from `out_techs.dat` (Step 3 must run first) and regenerates
`PES_data_remaining.dat` and `PES_data_remaining_wnd.dat`.

Re-run whenever new technologies are added or lifetimes change.

In [7]:
import Create_remaining as create_remaining

print('Regenerating remaining_years files ...')
create_remaining.run(
    out_techs_file     = str(TECHS_DIR / 'out_techs.dat'),
    remaining_file     = str(REMAINING_FILE),
    remaining_wnd_file = str(REMAINING_WND),
)

Regenerating remaining_years files ...
Loading lifetimes from: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs\out_techs.dat
  -> 689 technologies found
Written: c:\Users\matti\Desktop\EnergyScope-Quebec\projects\pathway\model\PES_data_remaining.dat
Written: c:\Users\matti\Desktop\EnergyScope-Quebec\projects\pathway\model\PES_data_remaining_wnd.dat


## Step 4 : Regenerate AGE file

Reads lifetimes from the snapshot dat files and regenerates
`PES_data_set_AGE_2020.dat`, which tells the model when the 2015_2020 stock
expires in each phase.

Re-run whenever a technology's lifetime changes in the snapshot model.

In [8]:
import Create_data_set_age_2020 as create_age

print("Regenerating AGE file ...")
create_age.run(
    out_techs_file = str(TECHS_DIR / "out_techs.dat"),
    output_file    = str(AGE_FILE),
)

Regenerating AGE file ...
Loading lifetimes from: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs\out_techs.dat
  -> 689 technologies found

Generated 4823 entries for 689 techs
Output written to: c:\Users\matti\Desktop\EnergyScope-Quebec\projects\pathway\model\PES_data_set_AGE_2020.dat


## Step 5 : Regenerate decom_allowed file

Reads lifetimes from `out_techs.dat` (Step 3 must run first) and regenerates
`PES_data_decom_allowed_2020.dat`.

Re-run whenever new technologies are added or lifetimes change.

In [9]:
import Create_decom_allowed as create_decom

print('Regenerating decom_allowed file ...')
create_decom.run(
    out_techs_file = str(TECHS_DIR / 'out_techs.dat'),
    output_file    = str(DECOM_FILE),
)

Regenerating decom_allowed file ...
Loading lifetimes from: c:\Users\matti\Desktop\EnergyScope-Quebec\shared\data\Techs\out_techs.dat
  -> 689 technologies found

Generated 9905 decom_allowed entries for 689 techs
Output written to: c:\Users\matti\Desktop\EnergyScope-Quebec\projects\pathway\model\PES_data_decom_allowed_2020.dat


## Step 6 : Regenerate years_active file

Reads lifetimes from `out_techs.dat` (Step 3 must run first) and regenerates
`PES_data_years_active.dat`, which encodes how many years each technology
installed in phase `p_inst` is still active during phase `p`.

Re-run whenever new technologies are added or lifetimes change.

In [ ]:
import Create_years_active as create_years_active

print('Regenerating years_active file ...')
create_years_active.run(
    out_techs_file     = str(TECHS_DIR / 'out_techs.dat'),
    years_active_file  = str(YEARS_ACTIVE_FILE),
)

## Summary

In [10]:
out_techs_path = TECHS_DIR / 'out_techs.dat'
if out_techs_path.exists():
    mtime  = datetime.fromtimestamp(out_techs_path.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
    n_lines = sum(1 for _ in out_techs_path.open())
    print(f'out_techs.dat   {mtime}   {n_lines:,} lines')
else:
    print('out_techs.dat not found -- run Steps 2 and 3.')

print()
for yr in YEAR_LABELS:
    tp = TECHS_DIR / f'QC_techs_{yr}.dat'
    ep = EUD_DIR   / f'QC_eud_{yr}.dat'
    t_ok = tp.exists()
    e_ok = ep.exists()
    print(f'  {yr}  techs: {"OK" if t_ok else "MISSING"}   eud: {"OK" if e_ok else "MISSING"}')

out_techs.dat   2026-05-19 14:38   4,685 lines

  2020  techs: OK   eud: OK
  2025  techs: OK   eud: OK
  2030  techs: OK   eud: OK
  2035  techs: OK   eud: OK
  2040  techs: OK   eud: OK
  2045  techs: OK   eud: OK
  2050  techs: OK   eud: OK
